In [1]:
!pip install beautifulsoup4 lxml pymorphy2 charset-normalizer

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 69.1 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=93b0b587a91bee2b3b78e0d1c2df1f55093d7031912f05dd127cc4318de8d675
  Stored in directory: /root/.cache/pip/wheels/1a/b0/8c/4b75c4116c31f83c8f9f047231251e13cc74481cca4a78a9ce
Successfully built docopt


In [41]:
from bs4 import BeautifulSoup, NavigableString

def search_token_contexts(xml_content, token, window=5):
    """
    Ищем все контексты, где встречается слово `token`.
    window — количество слов до и после слова.
    """
    # Преобразуем XML в структуру BeautifulSoup
    soup = BeautifulSoup(xml_content, 'lxml')

    # Находим все теги <w>, где обычно находятся слова
    words = soup.find_all('w')

    # Достаём текст из каждого тега <w>
    tokens = [w.get_text() for w in words]

    # Список для хранения найденных контекстов
    contexts = []

    # Приводим слово к нижнему регистру для нечувствительности к регистру
    token_lower = token.lower()

    # Перебираем все слова
    for i, word in enumerate(tokens):
        # Если найдено совпадение (без учёта регистра)
        if token_lower in word.lower():
            # Начало контекста (не меньше 0)
            start = max(i - window, 0)
            # Конец контекста (не больше длины текста)
            end = min(i + window + 1, len(tokens))

            # Получаем отрывок слов вокруг токена
            context = tokens[start:end]

            # Объединяем в одну строку и добавляем в список
            contexts.append(' '.join(context))

    # Возвращаем список контекстов
    return contexts

In [42]:
# Читаем XML-файл с кодировкой Windows-1251 (русский язык)
with open('venik_original.txt', encoding='windows-1251', errors='ignore') as f:
    content = f.read()

In [43]:
# Ищем слово 'веник' в тексте
results = search_token_contexts(content, 'веник')

In [44]:
# Выводим количество найденных контекстов и сами контексты
print(f"Найдено {len(results)} контекстов для токена 'веник':")
for c in results:
    print(c)

Найдено 541 контекстов для токена 'веник':
Приперся с веником Возьми березовый веник у Верки
Приперся с веником Возьми березовый веник у Верки Я и не
Я и не привыкла к венику У них везде в прихожей
там ну везде на газетках веники березовые разложены и сушатся А
разложены и сушатся А за вениками Веники перчатки совки мы частенько
и сушатся А за вениками Веники перчатки совки мы частенько покупаем
у нас и оклад небольшой Веник дубовый банный Эти веники разбрасываются
небольшой Веник дубовый банный Эти веники разбрасываются на голом мелководье Утром
пластиковые бутылки рыбаки легко находят веники ловушки Не стоит уподобляться им
иначе вы просто обменяетесь праздничными вениками и разойдетесь недовольные друг другом
недовольные друг другом А фирма веников как известно не вяжет Поэтому
не вяжет Поэтому огромный розовый веник может быть воспринят сегодня более
как чисто формальный жест Фирма веников не вяжет потрясающее музыкальное мастерство
Мог устроить спектакль начав сам веником и совком

Поиск контекстов по лемме

In [45]:
import inspect

# Patch для pymorphy2 под Python 3.11+
if not hasattr(inspect, 'getargspec'):
    def getargspec(func):
        from collections import namedtuple
        FullArgSpec = inspect.getfullargspec(func)
        ArgSpec = namedtuple('ArgSpec', 'args varargs keywords defaults')
        return ArgSpec(FullArgSpec.args, FullArgSpec.varargs, FullArgSpec.varkw, FullArgSpec.defaults)
    inspect.getargspec = getargspec

In [46]:
!pip install pymorphy2

In [47]:
from pymorphy2 import MorphAnalyzer

morph = MorphAnalyzer()

In [48]:
# Функция для лемматизации одного слова
def lemmatize_token(token):
    parsed_word = morph.parse(token)[0]  # Берём первый разбор слова
    return parsed_word.normal_form

In [49]:
# Функция для поиска контекста по лемме слова
def print_context_lemma(file_path, token, context_size=7):
    # Открываем XML-файл с кодировкой windows-1251
    with open(file_path, 'r', encoding='windows-1251') as file:
        content = file.read()

    # Разбираем XML с помощью BeautifulSoup
    soup = BeautifulSoup(content, 'lxml')

    # Получаем все теги <w> — они содержат слова
    w_tags = soup.find_all('w')

    # Множество уже напечатанных контекстов, чтобы не повторять
    printed_contexts = set()

    # Получаем лемму слова, которое мы ищем
    lemma_token = lemmatize_token(token)

    # Проходим по каждому слову в тексте
    for w_tag in w_tags:
        # Лемматизируем каждое слово внутри <w>
        lemmas = [lemmatize_token(word) for word in w_tag.text.split()]

        # Если нужная лемма есть среди них
        if lemma_token in lemmas:
            # Получаем индекс текущего тега среди всех дочерних элементов родителя
            w_index = list(w_tag.parent.children).index(w_tag)

            context_words = []

            # Идём назад и вперёд на context_size слов
            for i in range(max(0, w_index - context_size), min(len(w_tag.parent.contents), w_index + context_size + 1)):
                child = w_tag.parent.contents[i]

                # Пропускаем текст без тегов
                if isinstance(child, NavigableString):
                    continue

                # Добавляем текст внутри тега <w>
                if child.name == 'w':
                    context_words.append(child.text.strip())

            # Собираем контекст
            context_text = ' '.join(context_words)

            # Проверяем, был ли уже выведен такой контекст
            if context_text not in printed_contexts:
                print(f'Контекст для леммы "{lemma_token}": {context_text}')
                printed_contexts.add(context_text)

In [50]:
file_path = 'venik_original.txt'  # путь к XML-файлу

In [51]:
token = 'веники'  # это форма, но мы ищем по лемме "веник"

In [52]:
print_context_lemma(file_path, token, context_size=7)

Контекст для леммы "веник": Приперся с веником Возьми березовый веник
Контекст для леммы "веник": веником Возьми березовый веник у Верки Я
Контекст для леммы "веник": не привыкла к венику У них везде
Контекст для леммы "веник": везде на газетках веники березовые разложены и
Контекст для леммы "веник": сушатся А за вениками Веники перчатки совки
Контекст для леммы "веник": А за вениками Веники перчатки совки мы
Контекст для леммы "веник": и оклад небольшой Веник дубовый банный Эти
Контекст для леммы "веник": другом А фирма веников как известно не
Контекст для леммы "веник": самородок с кистью веником Анатолий Зверев вразумлявший
Контекст для леммы "веник": обрызгали водой с веника стабилизаторы Он бросает
Контекст для леммы "веник": Он бросает совок веник смеется бьет по
Контекст для леммы "веник": философы экстремалы любители веников с можжевельником и
Контекст для леммы "веник": затененной церкви шуршали вениками несколько старушек одна
Контекст для леммы "веник": семейной мудростью я

Лемматизируем все слова в тексте.

Ищем все вхождения заданного слова по лемме, а не по форме.

Показываем контекст вокруг каждого вхождения (по умолчанию 7 слов слева и справа).

Поиск контекстов по заданному семантическому тегу

In [53]:
# Функция для извлечения контекстов с нужным семантическим тегом из файла
def extract_contexts(file_path, semantic_tag, context_window=3):
    # Открываем файл с указанной кодировкой и читаем весь текст
    with open(file_path, 'r', encoding='windows-1251') as file:
        content = file.read()

    # Разбираем текст как HTML/XML с помощью BeautifulSoup
    soup = BeautifulSoup(content, 'html.parser')
    # Находим все теги <w> — это отдельные слова в тексте
    w_tags = soup.find_all('w')
    # Создаём список для хранения контекстов
    contexts = []

    # Проходим по каждому слову (тегу <w>) с индексом i
    for i in range(len(w_tags)):
        # Ищем внутри слова тег <ana>
        ana_tag = w_tags[i].find('ana')
        # Проверяем, что тег <ana> есть, и в его атрибутах есть 'sem'
        # и что нужный семантический тег содержится в атрибуте 'sem'
        if ana_tag and 'sem' in ana_tag.attrs and semantic_tag in ana_tag['sem']:
            # Определяем начало окна контекста (не меньше 0)
            context_start = max(0, i - context_window)
            # Определяем конец окна контекста (не больше длины списка слов)
            context_end = min(len(w_tags), i + context_window + 1)
            # Берём слова из окна контекста и получаем их текст
            context_words = [w_tags[j].get_text() for j in range(context_start, context_end)]
            # Объединяем слова в строку через пробел
            context_text = ' '.join(context_words)
            # Добавляем контекст в список
            contexts.append(context_text)

    # Возвращаем список уникальных контекстов (без повторов)
    return list(set(contexts))

In [54]:
# Путь к файлу с текстом
file_path = 'venik_original.txt'

In [55]:
# Семантический тег, который ищем в тексте
semantic_tag = 't:animal'

In [57]:
# Вызов функции с окном контекста 3 слова (слева и справа)
contexts = extract_contexts(file_path, semantic_tag, context_window=3)

In [58]:
# Выводим все найденные контексты на экран
for context in contexts:
    print(context)

мечами а на Волка пока с вениками
само собой молоденьким сучкам новый литератор Денискин
вибрациями от которых воробьи слетали с кустов
но растения веники уже отшумели и если
с курами и гусями на одном снимке
сказал Принц Яр Тур он у меня
в тепло Лейка совок веник и маленькие
веником по избушке комаров сгрудив младших ребят
установки самолеты конфеты Мишка одежду тарелки банные
были веник и совок и она делала
вездеходы шары зонды свиней полетные карты гвозди
вениками Веники перчатки совки мы частенько покупаем
листья для изгнания глистов Однако уже 17
охваченный пожаром он уже думал о ней
в момент выбулькивания рыбьего глаза иначе не
и вениками под мышками Накостылял бы он
было мусорное ведро совок и веник Огненными
и поднял цилиндр Уже почти в темноте
устроили скачки на овчарках а за ними
хмыкнул тихонько бородач Собака трепала его как
красные как вареные раки выскакивали в клубах
чтобы не горели тетерки а лишь подсыхали
опадают листья Антилопы овцебыки тоже уважают веники
зажав веник

Поиск по комбинации семантических тегов

In [59]:
def extract_contexts(file_path, semantic_tags, context_window=3):
    # Открываем файл с текстом в кодировке windows-1251
    with open(file_path, 'r', encoding='windows-1251') as file:
        content = file.read()

    # Используем BeautifulSoup для парсинга HTML/XML содержимого
    soup = BeautifulSoup(content, 'html.parser')

    # Находим все теги <w>
    w_tags = soup.find_all('w')

    contexts = []  # Список для хранения контекстов

    # Проходим по всем тегам <w>
    for i in range(len(w_tags)):
        # Ищем внутри каждого <w> тег <ana>
        ana_tag = w_tags[i].find('ana')

        # Проверяем, что тег <ana> существует и содержит все семантические теги из списка semantic_tags
        if ana_tag and all(tag in ana_tag.get('sem', '') for tag in semantic_tags):
            # Определяем границы окна контекста (слева и справа от текущего слова)
            context_start = max(0, i - context_window)
            context_end = min(len(w_tags), i + context_window + 1)

            # Собираем слова из окна контекста
            context_words = [w_tags[j].get_text() for j in range(context_start, context_end)]

            # Объединяем слова в строку
            context_text = ' '.join(context_words)

            # Добавляем контекст в список
            contexts.append(context_text)

    # Возвращаем уникальные контексты (удаляем повторы)
    return list(set(contexts))

In [60]:
# Семантические теги, которые мы ищем
semantic_tags = ['r:concr', 't:tool']

In [61]:
# Вызываем функцию для извлечения контекстов с окном в 3 слова с каждой стороны
contexts = extract_contexts(file_path, semantic_tags, context_window=3)

In [62]:
# Выводим все найденные контексты
for context in contexts:
    print(context)

в сени Под жерновами в истопке давно
Я люблю скрип саней по снегу баню
можжевельником и ледяной купели изрубила их на
в левую шайка Веники готовили заранее из
Ишь ты прямо веник махнул целую стаю
рюкзаки Вукол укладывал фонарики веревки щетки веники
петрушка зеленый лук веником репчатый косицами Там
одной рукой вроде веником меня отряхивает а
травы или мочалок лапти или шляпу из
правой рукой держа веник левой подметал осколки
представляющая каким концом веник в руки взять
голове дать хоть веником хоть мокрой тряпкой
в операторы водоразборного блока ходил с тряпкой
сметали с матрацев червей клопов и вшей
засалено в окошке веник торчит вот отчего
оружия ведро ботинок веник сумка книга Дома
по голове Веник метлу швабру Это в
яду или намыленного шнурка но не смог
ее кавалера в смокингах с помощью метрдотеля
как от мокрого веника На заранее отведенных
кипятком положил пока веник распаривать Он вынул
вместо кепочки дан веник а в левую
бегать за притормаживающими грузовиками катящими навстреч

Вывод результатов в ячейку или сохранение в файл

In [63]:
import pandas as pd

In [65]:
# Функция для извлечения информации из размеченного файла
def extract_info(file_path, context_window=3):
    # Открываем файл и читаем его содержимое с нужной кодировкой
    with open(file_path, 'r', encoding='windows-1251') as file:
        content = file.read()

    # Разбираем содержимое как HTML/XML
    soup = BeautifulSoup(content, 'html.parser')

    # Ищем все теги <w> — каждый тег <w> представляет собой слово
    w_tags = soup.find_all('w')

    # Создаём списки, чтобы сохранить информацию
    tokens = []             # исходные слова
    lemmas = []             # леммы (нормальные формы слов)
    grammatical_tags = []   # грамматические теги
    semantic_tags = []      # семантические теги
    contexts = []           # контексты (окружающие слова)

    # Проходим по всем словам в тексте
    for i in range(len(w_tags)):
        ana_tag = w_tags[i].find('ana')  # ищем тег <ana> внутри тега <w>
        if ana_tag:
            token = w_tags[i].get_text()  # само слово (токен)
            lemma = ana_tag.get('lex', '')  # лемма
            gram = ana_tag.get('gr', '')    # грамматическая информация
            sem = ana_tag.get('sem', '')    # семантическая метка

            # Определяем окно контекста: 3 слова слева и 3 справа
            context_start = max(0, i - context_window)
            context_end = min(len(w_tags), i + context_window + 1)
            context_words = [w_tags[j].get_text() for j in range(context_start, context_end)]
            context_text = ' '.join(context_words)  # объединяем слова в строку

            # Добавляем информацию в списки
            tokens.append(token)
            lemmas.append(lemma)
            grammatical_tags.append(gram)
            semantic_tags.append(sem)
            contexts.append(context_text)

    # Создаём таблицу (DataFrame) из всех списков
    df = pd.DataFrame({
        'Token': tokens,                     # колонка: слово
        'Lemma': lemmas,                     # колонка: лемма
        'Grammatical Tags': grammatical_tags,  # колонка: грам. тег
        'Semantic Tags': semantic_tags,        # колонка: сем. тег
        'Context': contexts                    # колонка: контекст
    })

    return df  # возвращаем таблицу

In [66]:
df = extract_info(file_path, context_window=3)
print(df)

          Token        Lemma                           Grammatical Tags  \
0      Приперся  Припираться       V,norm=praet,sg,indic,m,pf,med,indic   
1             с            с                                   PR,norm=   
2       веником        веник                       S,m,inan,norm=ins,sg   
3        Возьми        Взять                  V,pf,norm=sg,imper,2p,act   
4     березовый    березовый  A,norm=(nom,sg,m,plen|acc,sg,m,inan,plen)   
...         ...          ...                                        ...   
9029       тебя           ты               S-PRO,sg,anim,norm=(gen|acc)   
9030         не           не                                 PART,norm=   
9031      веник        веник              S,m,inan,norm=(nom,sg|acc,sg)   
9032          а            а                                 CONJ,norm=   
9033       коса         коса                       S,f,inan,norm=nom,sg   

                                          Semantic Tags  \
0                                       

In [67]:
# Сохраняем таблицу в CSV-файл с нужной кодировкой
df.to_csv('output.csv', index=False, encoding='windows-1251')

Здесь мы открываем XML-файл, извлекаем из него слова, их леммы, грамматические и семантические теги, а также контексты. Всё сохраняется в таблицу и экспортируется в файл output.csv.

Подсчет частотности токенов, лемм, тегов, комбинаций тегов

In [68]:
# Считает, сколько раз слово "веник" встречается в колонке "Token" (то, как написано в тексте)
token_count = df['Token'].str.count('веник').sum()

# Считает, сколько раз лемма "веник" (основная форма слова) встречается в колонке "Lemma"
lemma_count = df['Lemma'].str.count('веник').sum()

# Считает, сколько раз встречается тег "t:animal" в колонке семантических тегов
concr_tag_count = df['Semantic Tags'].str.count('t:animal').sum()

# Считает, сколько раз одновременно встречаются два тега: "r:concr" и "t:tool"
concr_tool_tag_count = df['Semantic Tags'].apply(lambda x: 'r:concr' in x and 't:tool' in x).sum()

print(f"\nПодсчет частотности:")
print(f"Token 'веник': {token_count}")
print(f"Lemma 'веник': {lemma_count}")
print(f"Tag 't:animal': {concr_tag_count}")
print(f"Tags 'r:concr t:tool': {concr_tool_tag_count}")


Подсчет частотности:
Token 'веник': 514
Lemma 'веник': 515
Tag 't:animal': 89
Tags 'r:concr t:tool': 931
